# Task 1 — Inconsistencies and duplicates
**CSEN1095 Data Engineering · W26** · Name: *<your name>* · ID: *<your id>*

Goal: turn `club_signups.csv` into one clean row per student per club. `club_signups.csv` is never modified; every fix is done in code and the result is written to `club_signups_clean.csv`.

| Section | Task step |
|---|---|
| 0 | Setup |
| A1 | Part A.1 – Look first |
| A2 | Part A.2 – Faculty, club, city |
| A3 | Part A.3 – Names and emails |
| A4 | Part A.4 – `fee_paid` boolean |
| A5 | Part A.5 – `signed_up_at` datetime |
| B1 | Part B.1 – Exact duplicates |
| B2 | Part B.2 – Same student, same club, submitted again |
| B3 | Part B.3 – Uniqueness assertion |
| Save | Save `club_signups_clean.csv` |
| C | Numbers used in the README (Part C) |

## 0. Setup
Only pandas is needed. The raw file is read once and kept untouched in `raw`; all work happens on a copy.

In [ ]:
import pandas as pd

raw = pd.read_csv("club_signups.csv", dtype=str)   # read everything as text: nothing is guessed for us
df = raw.copy()
rows_start = len(df)
print("rows at start:", rows_start)

## A1. Look first (Part A, step 1)
Shape, dtypes and `value_counts()` of every categorical column, **before** any fix, so the problems are visible.

In [ ]:
print("shape:", raw.shape)
print()
print(raw.dtypes)

In [ ]:
for col in ["faculty", "club", "city", "fee_paid"]:
    print(f"--- {col} ---")
    print(raw[col].value_counts(dropna=False))
    print()

In [ ]:
# Also look at the other columns: names/emails (spaces, case) and the two date formats
print(raw["full_name"].map(repr).value_counts().head(30))
print()
print(raw["email"].map(repr).value_counts().head(30))
print()
print(raw["signed_up_at"].sample(10, random_state=0).tolist())
print("missing values:\n", raw.isna().sum())

## A2. Faculty, club, city (Part A, step 2)
1. **Mechanical normalisation first**: collapse repeated spaces, strip, lower-case.
2. **Explicit dictionary** for what is left (e.g. *Media Engineering and Technology* → `MET`, *Soccer* → `Football`).
3. **Assert** that only the canonical values remain.

Unknown values are *not* silently kept: `map` returns `NaN` for a spelling that is not in the dictionary, and the assertion then fails loudly.

In [ ]:
def normalise(s: pd.Series) -> pd.Series:
    """strip, collapse inner whitespace, lower-case"""
    return s.str.replace(r"\s+", " ", regex=True).str.strip().str.lower()

FACULTY_MAP = {
    "met": "MET", "media engineering and technology": "MET", "media engineering & technology": "MET",
    "iet": "IET", "information engineering and technology": "IET", "information engineering & technology": "IET",
    "ems": "EMS", "engineering and materials science": "EMS", "engineering & materials science": "EMS",
    "pharmacy": "Pharmacy", "pharma": "Pharmacy",
    "management": "Management",
}
CLUB_MAP = {
    "robotics": "Robotics",
    "debate": "Debate", "debating": "Debate", "debate club": "Debate",
    "music": "Music", "music club": "Music",
    "football": "Football", "soccer": "Football", "soccer club": "Football", "football club": "Football",
    "chess": "Chess", "chess club": "Chess",
}
CITY_MAP = {
    "cairo": "Cairo",
    "giza": "Giza", "el giza": "Giza",
    "alexandria": "Alexandria", "alex": "Alexandria",
}

def fix_categories(frame: pd.DataFrame) -> pd.DataFrame:
    frame = frame.copy()
    frame["faculty"] = normalise(frame["faculty"]).map(FACULTY_MAP)
    frame["club"]    = normalise(frame["club"]).map(CLUB_MAP)
    frame["city"]    = normalise(frame["city"]).map(CITY_MAP)
    return frame

df = fix_categories(df)

CANON = {
    "faculty": {"MET", "IET", "EMS", "Pharmacy", "Management"},
    "club":    {"Robotics", "Debate", "Music", "Football", "Chess"},
    "city":    {"Cairo", "Giza", "Alexandria"},
}
for col, allowed in CANON.items():
    assert df[col].notna().all(), f"{col}: a value was not in the dictionary"
    assert set(df[col].unique()) <= allowed, f"{col}: non-canonical value left"
    print(col, "->", sorted(df[col].unique()))

## A3. Names and emails (Part A, step 3)
Names: no extra spaces anywhere, same capitalisation for everyone (Title Case). Emails: trimmed and lower-case.
The `student_id` is **not** used to fix names, it is only trimmed.

In [ ]:
df["full_name"] = (df["full_name"].str.replace(r"\s+", " ", regex=True)
                                  .str.strip()
                                  .str.title())
df["email"] = df["email"].str.strip().str.lower()
df["student_id"] = df["student_id"].str.strip()

assert not df["full_name"].str.contains(r"^\s|\s$|\s{2,}").any()
assert (df["email"] == df["email"].str.strip().str.lower()).all()
print(df["full_name"].value_counts())

## A4. `fee_paid` → boolean (Part A, step 4)
Explicit dictionary; an unknown spelling becomes `NaN` and trips the assertion instead of being silently turned into `True`.

In [ ]:
FEE_MAP = {"yes": True, "y": True, "true": True, "1": True,
           "no": False, "n": False, "false": False, "0": False}
df["fee_paid"] = df["fee_paid"].str.strip().str.lower().map(FEE_MAP)
assert df["fee_paid"].notna().all(), "unknown fee_paid spelling"
df["fee_paid"] = df["fee_paid"].astype(bool)
print(df["fee_paid"].dtype, df["fee_paid"].value_counts().to_dict())

## A5. `signed_up_at` → one datetime column (Part A, step 5)
Two formats are mixed: ISO `YYYY-MM-DD HH:MM` and `DD/MM/YYYY HH:MM`.
Day vs month: in the slash format the **first** number is the day (values such as `15/09/2026`, `17/09/2026`, `18/09/2026` cannot be months, and the ISO rows show all submissions are in September 2026).
Each format is parsed with its own **explicit** format string, so no guessing happens, and the time of day is **kept** (needed later to pick the latest submission).

In [ ]:
s = df["signed_up_at"].str.strip()
is_iso   = s.str.match(r"^\d{4}-\d{2}-\d{2} ")
is_slash = s.str.match(r"^\d{2}/\d{2}/\d{4} ")
print("ISO rows:", is_iso.sum(), "| slash rows:", is_slash.sum(), "| unmatched:", (~(is_iso | is_slash)).sum())
assert (is_iso | is_slash).all()

# evidence for day-first: the first number in slash dates goes above 12
print("max first number in slash dates:", s[is_slash].str[:2].astype(int).max(),
      "| max second number:", s[is_slash].str[3:5].astype(int).max())

parsed = pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns]")
parsed[is_iso]   = pd.to_datetime(s[is_iso],   format="%Y-%m-%d %H:%M")
parsed[is_slash] = pd.to_datetime(s[is_slash], format="%d/%m/%Y %H:%M")
df["signed_up_at"] = parsed

assert df["signed_up_at"].notna().all()
assert df["signed_up_at"].between("2026-09-01", "2026-09-30").all()   # all September => day/month the right way round
print(df["signed_up_at"].dtype, df["signed_up_at"].min(), "->", df["signed_up_at"].max())

## B1. Exact duplicates (Part B, step 1)
Same form submission twice = all columns identical. This is checked **after** Part A so that spelling differences cannot hide a duplicate. Count, then keep the first copy.

In [ ]:
n_before = len(df)
n_exact = int(df.duplicated(keep="first").sum())
print("exact duplicate rows:", n_exact)
print(df[df.duplicated(keep=False)].sort_values("signup_id")[["signup_id", "student_id", "club", "signed_up_at"]])

df = df.drop_duplicates(keep="first")
assert len(df) == n_before - n_exact
rows_after_exact = len(df)
print("rows left:", rows_after_exact)

## B2. Same sign-up submitted again (Part B, step 2)
Same `student_id` + `club`, different `signup_id`. Keep the **latest** submission (sort by the full timestamp, keep the last).
Key = `student_id` (not the name): two different students called *Mohamed Adel* have different ids, so they are never merged.

In [ ]:
df = df.sort_values(["signed_up_at", "signup_id"], kind="stable")
n_before = len(df)
dupes = df[df.duplicated(["student_id", "club"], keep=False)].sort_values(["student_id", "club", "signed_up_at"])
print(dupes[["signup_id", "student_id", "full_name", "club", "fee_paid", "signed_up_at"]])

df = df.drop_duplicates(subset=["student_id", "club"], keep="last")
n_same_signup = n_before - len(df)
rows_final = len(df)
print("rows removed:", n_same_signup, "| rows left:", rows_final)

## B3. Check (Part B, step 3)
`student_id + club` must be unique.

In [ ]:
assert not df.duplicated(subset=["student_id", "club"]).any()
assert rows_start - n_exact - n_same_signup == rows_final
print("OK: student_id + club is unique;", rows_start, "-", n_exact, "-", n_same_signup, "=", rows_final)

## Save
Restore the original order of the form (by `signup_id`), reset the index, and write the result.

In [ ]:
clean = df.sort_values("signup_id").reset_index(drop=True)
clean.to_csv("club_signups_clean.csv", index=False)
print(clean.dtypes)
clean

## C. Numbers for the README (Part C)
Evidence for the *order of steps* question: what would have happened if duplicates were removed **before** fixing the spellings, working on the raw strings?
The helper below removes exact copies, then dedupes `student_id + club` on the raw text.

In [ ]:
r = raw.copy()
raw_exact   = int(r.duplicated().sum())
r1          = r.drop_duplicates()
raw_same    = int(r1.duplicated(["student_id", "club"], keep="last").sum())
print("WRONG ORDER (raw spellings): exact =", raw_exact, "| same-signup =", raw_same, "| left =", len(r1) - raw_same)
print("RIGHT ORDER (after cleaning): exact =", n_exact, "| same-signup =", n_same_signup, "| left =", rows_final)
print("same-signup duplicates missed by wrong order:", n_same_signup - raw_same)

# two different students with the same name
names = clean.groupby("full_name")["student_id"].nunique()
print(names[names > 1])